In [100]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
np.random.seed(42)

In [101]:
#creamos el mercado
def mercado(S0, T, r, sigma, num_simulations, num_steps):
    np.random.seed(42)
    dt = T / num_steps
    drift = (r - 0.5 * sigma**2) * dt
    diffusion = sigma * np.sqrt(dt)

    Z = np.random.normal(0, 1, (num_steps, num_simulations))
    log_returns = drift + diffusion * Z

    # Acumular sobre el tiempo
    price_path = S0 * np.exp(np.cumsum(log_returns, axis=0))
    return price_path


In [102]:
#asian pricer
from scipy.stats import gmean

def arit_asian(price_path, K,r,T, option_type= 'Call'):
    price = 0
    path_mean = np.mean(price_path, axis=0)
    if option_type == 'Call':
        price = np.maximum(0, path_mean - K)
    elif option_type == 'Put':
        price = np.maximum(0, K-path_mean )
    else: print('Escribe bien pendejo')
    return np.exp(-r * T) * price

def geo_asian(price_path, K,r,T, option_type= 'Call'):
    price = 0
    path_mean = gmean(price_path, axis=0)
    if option_type == 'Call':
        price = np.maximum(0, path_mean - K)
    elif option_type == 'Put':
        price = np.maximum(0, K-path_mean )
    else: print('Escribe bien pendejo')
    return np.exp(-r * T) * price

In [103]:
#black scholes asian
def black_scholes_asian(S0, K, r, sigma, T, option_type="Call"):
    # d1 y d2
    Sigma_A = sigma/np.sqrt(3)

    b_A = (1/2) * (r-(Sigma_A**2/2))

    d1 = (np.log(S0 / K) + (b_A + 0.5 * Sigma_A**2) * T) / (Sigma_A * np.sqrt(T))
    d2 = d1 - Sigma_A * np.sqrt(T)

    if option_type == "Call":
        price = S0*np.exp(b_A-r) * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    elif option_type == "Put":
        price = K * np.exp(-r * T) * norm.cdf(-d2) - S0*np.exp(b_A - r) * norm.cdf(-d1)
    return price

In [104]:
#parametrods
K = 100
So = 100
sigma = .20
r = .05
T = 1

price_geo =black_scholes_asian(So, K, r, sigma, T, 'Call')
print(f'Precio del activo geometrico asiático es: {price_geo}')


Precio del activo geometrico asiático es: 5.546818633789201


decimos que...
$$y = \text{Precio de Opción Asiática}$$
$$x = \text{Precio de Opción Geométrico}$$


In [105]:
MERCADO = mercado(So, T, r, sigma, num_simulations=10, num_steps=252)

In [106]:
p_a=arit_asian(MERCADO, K,r,T)
p_g = geo_asian(MERCADO,K,r,T)


In [107]:
def monte_carlo_control_variable(S0, K, r, sigma, T, num_sims, num_steps , option_type='Call'):
    price_paths = mercado(S0, T, r, sigma, num_simulations=num_sims, num_steps=num_steps)

    Y = arit_asian(price_paths, K, r, T, option_type)
    X = geo_asian(price_paths, K, r, T, option_type)

    mu_X = black_scholes_asian(S0, K, r, sigma, T, option_type)


    cov_matrix = np.cov(Y, X)
    cov_YX = cov_matrix[0, 1]
    var_X = cov_matrix[1, 1]
    b_hat = cov_YX / var_X

    #ajustados por variable de control
    Y_cv = Y - b_hat * (X - mu_X)

    mc_price = np.mean(Y)

    cv_price = np.mean(Y_cv)
    cv_se = np.std(Y_cv, ddof=1) / np.sqrt(num_sims)

    variance_reduction = (1 - (np.var(Y_cv, ddof=1) / np.var(Y, ddof=1))) * 100

    resultados = {
        'b_estimada': b_hat,
        'Precio Mont. Arit': mc_price,
        'Precio sol cerrada': mu_X,
        'precio_Variable_Control': cv_price,
        'error_estandar_CV': cv_se,
        'reduccion_varianza_pct': variance_reduction}

    resultados = pd.DataFrame(resultados.values(), index= resultados.items())
    return resultados

In [109]:
print(monte_carlo_control_variable(So, K, r, sigma, 1, 100, 252))

                                           0
b_estimada              1.031623    1.031623
Precio Mont. Arit       5.915567    5.915567
Precio sol cerrada      5.546819    5.546819
precio_Variable_Control 5.748500    5.748500
error_estandar_CV       0.021956    0.021956
reduccion_varianza_pct  99.918787  99.918787
